# Swin feature ablation for Solafune

The Swin backbone, fold, samples, seed, and training settings are fixed. Only input features are changed.

In [ ]:
from argparse import Namespace
from pathlib import Path
import subprocess
import sys
import torch

REPO = Path('/kaggle/working/solafune-nowcast')
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/shionsuio/solafune-nowcast.git', str(REPO)], check=True)

sys.path.insert(0, str(REPO / 'src'))

from kaggle_setup import ensure_kaggle_workspace
from experiments.swin_ablation import run

ensure_kaggle_workspace(Path('/kaggle/working'))
assert torch.cuda.is_available(), 'Kaggle Notebook settingsでGPUを有効にしてください'
print(torch.cuda.get_device_name(0))

# Optional read-only cache source. Newly computed stats are always saved under /kaggle/working.
STATS_DATASET = Path('/kaggle/input/datasets/suioshion/solafune-stat')
BAND_STATS_ROOT = str(STATS_DATASET) if STATS_DATASET.exists() else None

args = Namespace(
    root='/kaggle/working',
    fold=0,
    train_rows=1800,
    validation_rows=900,
    stats_rows_per_satellite=300,
    epochs=3,
    batch_size=8,
    seed=42,
    band_stats_root=BAND_STATS_ROOT,
    output='outputs/swin_ablation/results.csv',
)

result_path = run(args)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

results = pd.read_csv(result_path).sort_values('best_validation_rmse')
display(results)
ax = results.sort_values('best_validation_rmse', ascending=False).plot.barh(
    x='experiment', y='best_validation_rmse', figsize=(9, 5), legend=False
)
ax.set_xlabel('Validation RMSE (lower is better)')
ax.set_ylabel('')
plt.tight_layout()
plt.show()
